In [ ]:
import pandas as pd

## Source of Truth

In [ ]:
truthdf=pd.read_csv('source_of_truth.csv')

In [ ]:
print(truthdf.columns)

In [ ]:
truthdf.count()

In [ ]:
print(truthdf.dtypes)

In [ ]:
# 1. Identify all float columns
float_cols = truthdf.select_dtypes(include=['float64']).columns

# 2. Use .loc to set values directly into the original dataframe
# This solves the SettingWithCopyWarning
truthdf.loc[:, float_cols] = truthdf[float_cols].fillna(0).astype(int)

# 3. Specifically fix the 'HCC (Y/N)' column since it was an object type
if 'HCC (Y/N)' in truthdf.columns:
    truthdf.loc[:, 'HCC (Y/N)'] = (
        pd.to_numeric(truthdf['HCC (Y/N)'], errors='coerce')
        .fillna(0)
        .astype(int)
    )

print("Data types after conversion:")
print(truthdf.dtypes)

### Filter cols

In [ ]:
truthdf=truthdf[['Random ID', 'DEATH (Y=1/N=0)',
       'Death Date (DD/MM/YYYY)', 'Transplant (Y/N)', 'Date of Transplant',
       'Variceal bleed (Y/N)', 'First Variceal bleed Date', 'Ascites (Y/N)',
       'First Ascites Date', 'SBP (Y/N)', 'First SBP Date', 'HE (Y/N)',
       'First HE Date', 'HCC (Y/N)', 'First HCC Date', 'PVT (Y/N)',
       'First PVT Date', 'TIPS (Y/N)', 'First TIPS Date', 'Sepsis (Y/N)',
       'First Sepsis Date', 'Hepatorenal syndrome (Y/N)',
       'First Hepatorenal syndrome Date', 'Hyponatremia (Y/N)',
       'First Hyponatremia Date', 'ACLF (Y/N)', 'First ACLF Date']]

In [ ]:
truthdf.to_csv("cleaned_truth.csv", index=False)

## Radiology Report

In [ ]:
radiodf=pd.read_csv('radio.csv')

### Filter Rows
this is based on my observation but only reports starting w the words history or original contain radio reports

In [ ]:
radiodf = radiodf[radiodf['Text'].str.match(r'^(history|original)', case=False, na=False)]
radiodf

### Clean Text using RegEx
based on obervation as well (not 100% accurately cleaning off names)

In [ ]:
import re
import pandas as pd

# Compile regex once for performance
RE_PR_UPTO_CONC = re.compile(
    r'(?i)(performing\s*radiographer\s*:)\s*[a-z\s-]*?(?=\bconclusion\b)'
)
RE_PR_FALLBACK = re.compile(
    r'(?i)(performing\s*radiographer\s*:)\s*[a-z]+(?:\s+[a-z]+)*'
)

RE_FINALISED = re.compile(
    r'''(?isx)
    (finalised\s*by\s*:)     # \1 = the label we keep
    \s*
    (                        # \2 = the span to remove
        .*?                  # lazily consume anything (across lines)
        (?=\bfinalised\b|\breported\s*:)    # stop BEFORE next 'finalised' OR 'reported:'
        |
        .*?\Z                # OR consume to end of string if neither appears
    )
    '''
)

RE_REPORTED = re.compile(
    r'(?i)(reported\s*by\s*:)\s*[a-z]+(?:\s+[a-z]+)*'
)


_LABELS = r'technique|procedure|findings|conclusion|impression|reported|report|guidance'

# Operators: ... up to the next punctuation ('.', ':', '。', '：') or next known label (optional colon)
RE_OPERATOR_TO_NEXT = re.compile(
    r'''(?isx)
    (operator(?:\s*\(s\)|s)?\s*:)   # \1 = "operator:", "operators:", "operator(s):", "operator (s):"
    \s*
    .*?                             # scrub non-greedily across lines
    (?=                             # stop BEFORE one of:
        \b(?:technique|procedure|findings|conclusion|impression|reported|report|guidance)\b\s*:?
        |
        \Z                          # or end of string
    )
    '''
)

# NEW: remove 'Dr' + next two words
RE_DR_NEXT2 = re.compile(
    r'''(?ix)
    \bdr\.?\s+
    [A-Za-z]+(?:[.'-][A-Za-z]+)*
    \s+
    [A-Za-z]+(?:[.'-][A-Za-z]+)*
    '''
)

def scrub_names(text: str) -> str:
    if pd.isna(text):
        return text
    s = str(text)

    # 1) Performing radiographer: remove up to "conclusion" (if present)
    s = RE_PR_UPTO_CONC.sub(r'', s)

    # 2) Performing radiographer: fallback -> remove letters/spaces until a non-letter
    s = RE_PR_FALLBACK.sub(r'', s)

    # 3) Finalised by: remove everything until next 'finalised' (or end)
    s = RE_FINALISED.sub(r'', s)

    # 4) Reported by: remove letters/spaces until a non-letter
    s = RE_REPORTED.sub(r'', s)

    # 5) Operator(s): ... up to Technique:
    s = RE_OPERATOR_TO_NEXT.sub(r'', s)

    # 6) Dr + 2 words (last local scrub)
    s = RE_DR_NEXT2.sub(r'', s)

    return s

In [ ]:
radiodf.loc[:, 'Text'] = radiodf['Text'].map(scrub_names)

#### Export cleaned radiology dataset

In [ ]:
radiodf=radiodf.reset_index(drop=True)
radiodf.to_csv("cleaned_radio.csv", index=False)

## General Lab

based on the information given in prompt.docx, these are the 3 labels that can be affected by General Lab results: ascites, HRS, HE

In [ ]:
lab1df=pd.read_csv('lab1.csv',low_memory=False)
lab2df=pd.read_csv('lab2.csv',low_memory=False)

In [ ]:
lab1df = lab1df[lab1df['Random ID'].isin(df1['Random ID'])]
cleaned_lab1=lab1df[['Random ID', 'Lab Test Code','Lab Resulted Order Test Description', 'Reference Ranges','Result Value', 'Reported Date']]
cleaned_lab1

In [ ]:
lab2df = lab2df[lab2df['Random ID'].isin(df1['Random ID'])]
cleaned_lab2=lab2df[['Random ID', 'Lab Resulted Order Test Description', 'Reference Ranges','Result Value']]
cleaned_lab2

In [ ]:
cleaned_lab = pd.concat([cleaned_lab1, cleaned_lab2], axis=0, ignore_index=True, sort=False)
cleaned_lab

In [ ]:
required_tests = [
    'CREATININE', 
    'SODIUM', 
    'ALBUMIN', 
    'BILIRUBIN,TOTAL', 
    'PROTHROMBIN TIME', 
    'AMMONIA,PLASMA', 
    'WBC', 
    'ALPHAFOETO PROTEIN', 
    'POLYMORPHONUCLEAR LEUCOCYTES',
    'NEUTROPHILS',
    'CELL CT FLUID',
    'FLUID DIFF CT',
    'FLUID DIFFERENTIAL COUNT'
]

# 1. Standardise to uppercase and strip hidden spaces
cleaned_lab['Lab Resulted Order Test Description'] = cleaned_lab['Lab Resulted Order Test Description'].str.upper().str.strip()

# 2. Exact match only (isin performs an '==' check for every item in the list)
cleaned_lab = cleaned_lab[cleaned_lab['Lab Resulted Order Test Description'].isin(required_tests)]

In [ ]:
cleaned_lab

#### Export cleaned general lab dataset

In [ ]:
cleaned_lab.to_csv("cleaned_lab.csv", index=False)

### Filter Gen Lab Rows by amount of substance found relative to reference range
did not use this but keeping as archive because im not sure how the general lab is being used for diagnosis

In [ ]:
def filter_alarming_rows(row):
    test = str(row['Lab Resulted Order Test Description']).upper()
    try:
        # Convert value to float for comparison
        val = float(row['Result Value'])
    except:
        # If it's a string like "Positive", keep it for the LLM to interpret
        return True 

    # --- CATEGORY: ALWAYS KEEP (Critical for Calculation/Trends) ---
    if any(x in test for x in ['CREATININE', 'BILIRUBIN', 'PROTHROMBIN', 'AMMONIA', 'ALPHAFOETO', 'LEUCOCYTES']):
        return True

    # --- CATEGORY: CONDITIONAL KEEP (Only if abnormal/borderline) ---
    if 'SODIUM' in test:
        return val < 130  # Keep if even slightly low (135 and below)
    
    if 'ALBUMIN' in test:
        return val < 36   # Keep if even slightly low (35 and below)
    
    if 'WBC' in test:
        return val > 11.0 or val < 4.0  # Only keep if signs of infection or bone marrow stress
    
    return False

# Apply the filter
df_filtered = cleaned_lab[cleaned_lab.apply(filter_alarming_rows, axis=1)]

In [ ]:
df_filtered 

## Discharge Summary

In [ ]:
df_discharge=pd.read_csv('discharge_summary.csv',low_memory=False)

### Clean Rows n Cols

In [ ]:
Document_Items=['SHS_DscSum_DischargeSummary_TXT','SHS_DscSum_Procedure_TXT','SHS_DscSum_MedicationPrescribed_TXT']
# Retain only rows where the description is in your specified array
df_discharge = df_discharge[df_discharge['Document Item Description'].isin(Document_Items)]


In [ ]:
cleaned_discharge=df_discharge[['Random ID', 'Visit Date (YYYYMMDD)','Full text']]

is_number_mask=pd.to_numeric(cleaned_discharge['Full text'], errors='coerce').notna()
text_str=cleaned_discharge['Full text'].astype(str).str.strip()
is_dash_or_empty=text_str.isin(['-',''])
cleaned_discharge = cleaned_discharge[~is_number_mask & ~is_dash_or_empty]
cleaned_discharge

In [ ]:
cleaned_discharge.loc[:, 'Full text'] = cleaned_discharge['Full text'].map(scrub_names)

#### Export cleaned discharge summary dataset

In [ ]:
cleaned_discharge.to_csv("cleaned_discharge_summary.csv", index=False)

## Outpatient Summary (gastro notes)

In [ ]:
df_outpatient=pd.read_csv('gastro_notes.csv',low_memory=False)

### Clean Rows n Cols

In [ ]:
cleaned_outpatient = df_outpatient[(df_outpatient['Document Item Description'] == 'Clinical Note') | 
                            (df_outpatient['Document Item Description'] == 'SGH_Consult_ExecSum_TXT')]
cleaned_outpatient

In [ ]:
cleaned_outpatient.to_csv("cleaned_outpatient_summary.csv", index=False)

## Outpatient Summary (gastro notes)

In [ ]:
df_endoscope=pd.read_csv('endoscope.csv',low_memory=False)

### Clean Rows n Cols

In [ ]:
cleaned_endoscope = df_endoscope[['Random ID', 'Procedure Start Date','Summary of Procedure']]
# 1. Remove exact duplicate rows
cleaned_endoscope = cleaned_endoscope.drop_duplicates()

# 2. Remove rows where the data is NA/Null
cleaned_endoscope = cleaned_endoscope.dropna()

# 3. Remove rows where the length is less than 15 characters
# Replace 'Snippet' with the actual column name you want to check (e.g., 'Discharge_Entry')
target_col = 'Summary of Procedure' 
cleaned_endoscope = cleaned_endoscope[cleaned_endoscope[target_col].astype(str).str.len() >= 15]

# Reset the index to keep it clean
cleaned_endoscope = cleaned_endoscope.reset_index(drop=True)
cleaned_endoscope

In [ ]:
cleaned_endoscope.to_csv("cleaned_endoscope.csv", index=False)

### Merged df of all Notes
cannot be saved and need to rerun every round u want to use because it will exceed excel word limit per cell

In [ ]:
import pandas as pd
df_imaging=pd.read_csv('cleaned_radio.csv')
df_labs=pd.read_csv('cleaned_lab.csv')
df_truth=pd.read_csv('cleaned_truth.csv')
df_discharge=pd.read_csv('cleaned_discharge_summary.csv')
df_outpatient=pd.read_csv('cleaned_outpatient_summary.csv')
df_endoscope=pd.read_csv('cleaned_endoscope.csv')

In [ ]:
df_patients=(df_truth.copy()[['Random ID']]).dropna(subset=['Random ID'])
df_patients

import pandas as pd

def process_labs(df_labs):
    """Clean, pivot, and flatten lab data."""
    df = df_labs.copy()
    # Parse date safely
    df['timestamp'] = pd.to_datetime(df['Reported Date'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort to preserve chronological order in concatenation
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline for the LLM
    df['Lab_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + " " +
        df['Lab Resulted Order Test Description'].astype(str) + ": " +
        df['Result Value'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Lab_Entry'].apply(' | '.join).reset_index()

def process_imaging(df_imaging):
    df = df_imaging.copy()
    # Remove dayfirst=True to let pandas handle the YYYY-MM-DD format correctly
    df['timestamp'] = pd.to_datetime(df['Performed Date Time'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    df = df.sort_values(['Random ID', 'timestamp'])
    df['Img_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Text'].astype(str)

    return df.groupby('Random ID', sort=False)['Img_Entry'].apply(' | '.join).reset_index()


def process_discharge(df_discharge):
    """
    Clean and flatten discharge summaries.

    Expected columns:
      - 'Visit Date (YYYYMMDD)' : date in YYYYMMDD format (string or int)
      - 'Full text'             : discharge note content
      - 'Random ID'             : patient identifier
    """
    df = df_discharge.copy()

    # Parse YYYYMMDD robustly (works if the column is str or int)
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort so concatenation is in chronological order
    df = df.sort_values(['Random ID', 'timestamp'])
    # Build entry text
    df['Discharge_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Full text'].astype(str)

    # Group per patient
    return df.groupby('Random ID', sort=False)['Discharge_Entry'].apply(' | '.join).reset_index()
def process_outpatient(df_outpatient):
    """Clean and flatten outpatient clinical notes."""
    df = df_outpatient.copy()
    
    # Filter for relevant clinical notes only
    valid_docs = ['Clinical Note', 'SGH_Consult_ExecSum_TXT']
    df = df[df['Document Item Description'].isin(valid_docs)]
    
    # Parse YYYYMMDD date format
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort chronologically
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline
    df['Outpatient_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + 
        df['Document Item Description'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Outpatient_Entry'].apply(' | '.join).reset_index()

def process_endoscopy(df_endoscope):
    """Clean and flatten endoscopy reports."""
    df = df_endoscope.copy()
    
    # 1. Drop exact duplicates, NAs, and short text (< 15 chars)
    df = df.drop_duplicates()
    df = df.dropna(subset=['Summary of Procedure', 'Procedure Start Date'])
    df = df[df['Summary of Procedure'].astype(str).str.len() >= 15]

    # 2. Parse DD/MM/YYYY date safely
    df['timestamp'] = pd.to_datetime(df['Procedure Start Date'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # 3. Sort and create entry string
    df = df.sort_values(['Random ID', 'timestamp'])
    df['Endo_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Summary of Procedure'].astype(str)

    # 4. Group per patient
    return df.groupby('Random ID', sort=False)['Endo_Entry'].apply(' | '.join).reset_index()

def create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope):
    """Merges all sources including endoscopy."""
    labs_processed = process_labs(df_labs)
    imaging_processed = process_imaging(df_imaging)
    discharge_processed = process_discharge(df_discharge)
    outpatient_processed = process_outpatient(df_outpatient)
    endo_processed = process_endoscopy(df_endoscope) # New source

    master = (
        df_patients[['Random ID']]
        .merge(labs_processed, on='Random ID', how='left')
        .merge(imaging_processed, on='Random ID', how='left')
        .merge(discharge_processed, on='Random ID', how='left')
        .merge(outpatient_processed, on='Random ID', how='left')
        .merge(endo_processed, on='Random ID', how='left') # New merge
    )

    # Keep row if at least one narrative text source exists
    text_cols = ['Img_Entry', 'Discharge_Entry', 'Outpatient_Entry', 'Endo_Entry']
    master = master.dropna(subset=text_cols, how='all')

    return master.reset_index(drop=True)

# UPDATED EXECUTION
final_summary = create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope)

final_summary

#### Get all Random IDs with at least 1 non NULL source

In [ ]:
# Save only the unique clean IDs to a CSV
final_summary[['Random ID']].drop_duplicates().to_csv('valid_ids.csv', index=False)
print(f"Saved {len(final_summary['Random ID'].unique())} IDs to valid_ids.csv")

In [ ]:
# 1. Convert both to actual integers first to strip decimals and spaces
# This creates a 'clean' set of numbers from your summary
final_ids_numeric = pd.to_numeric(final_summary['Random ID'], errors='coerce').dropna().astype(int).unique()

# 2. Create a numeric version of df_truth IDs for the comparison
df_truth_numeric = pd.to_numeric(df_truth['Random ID'], errors='coerce')

# 3. Filter df_truth based on the numeric match
df_truth = df_truth[df_truth_numeric.isin(final_ids_numeric)].copy()

# 4. Now that we have the right rows, sync the string format to match final_summary
df_truth['Random ID'] = df_truth_numeric.loc[df_truth.index].astype(int).astype(str)

print(f"Rows in final_summary: {len(final_summary)}")
print(f"Rows now in df_truth: {len(df_truth)}")



### Positive and Negative per Sample in Final_Summary

In [ ]:
def get_final_truth_counts(df_truth, final_summary):
    # 1. Deep Clean IDs: Remove .0, hidden spaces, and non-breaking characters
    def deep_clean_ids(series):
        return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.replace(r'\xa0', '', regex=True)

    summary_ids = deep_clean_ids(final_summary['Random ID']).unique()
    truth_copy = df_truth.copy()
    truth_copy['Random ID'] = deep_clean_ids(truth_copy['Random ID'])
    
    # 2. Filter for Overlap
    filtered_truth = truth_copy[truth_copy['Random ID'].isin(summary_ids)].copy()
    
    if filtered_truth.empty:
        print("!!! Still no overlap found between IDs.")
        return pd.DataFrame()

    # 3. Target medical columns
    target_cols = ['Ascites (Y/N)', 'Variceal bleed (Y/N)', 'HE (Y/N)', 'SBP (Y/N)', 'HCC (Y/N)', 'PVT (Y/N)', 'TIPS (Y/N)']
    
    # 4. Standardize 1.0/0.0 into clean integers
    for col in target_cols:
        if col in filtered_truth.columns:
            filtered_truth[col] = pd.to_numeric(filtered_truth[col], errors='coerce').fillna(0).astype(int)

    # 5. Calculate counts
    # .value_counts().T gives us a row for each condition and columns for 0 and 1
    stats = filtered_truth[target_cols].apply(lambda x: x.value_counts()).T.fillna(0).astype(int)
    
    # Rename columns safely
    stats = stats.rename(columns={0: 'Count_0', 1: 'Count_1'})
    
    # 6. ADD TOTAL SAMPLES COL
    # This sums the 0s and 1s for every row
    stats['Total_Samples'] = stats['Count_0'] + stats['Count_1']

    print(f"Overlap size: {len(filtered_truth)} unique patients.")
    return stats

# Run and view results
final_truth_stats = get_final_truth_counts(df_truth, final_summary)
print(final_truth_stats)